# Hair Extraction & Visualization - Google Colab
Extract and display hair region from face parsing segmentation.
Uses Google Drive for model and image storage.

## 1. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print("✓ Google Drive mounted successfully")

## 2. Setup & Dependencies

In [ ]:
import torch
import numpy as np
import cv2
import matplotlib.pyplot as plt
import os
from PIL import Image
from transformers import (
    SegformerImageProcessor,
    SegformerForSemanticSegmentation,
)

# Set device
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"✓ Using device: {device}")
if device == "cuda":
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")

## 3. Load Model from Google Drive

In [ ]:
# CONFIGURATION - Update these paths to match your Google Drive structure
# Example structure:
# My Drive/
#   ├── face_parsing/  (model folder)
#   └── images/
#       └── side.png

model_folder_path = "/content/drive/MyDrive/face_parsing"
image_path = "/content/drive/MyDrive/images/side.png"

print(f"Model path: {model_folder_path}")
print(f"Image path: {image_path}")
print()

# Check if paths exist
if os.path.exists(model_folder_path):
    print(f"✓ Model folder found")
    files = os.listdir(model_folder_path)
    print(f"  Contents: {files[:5]}..." if len(files) > 5 else f"  Contents: {files}")
else:
    print(f"✗ Model folder not found at: {model_folder_path}")
    print("  Please update model_folder_path in cell above")
    print()

if os.path.exists(image_path):
    print(f"✓ Image file found")
else:
    print(f"✗ Image file not found at: {image_path}")
    print("  Please update image_path in cell above")

## 4. Load Face Parsing Model from Google Drive

In [ ]:
try:
    print("Loading model from Google Drive...")
    processor = SegformerImageProcessor.from_pretrained(model_folder_path)
    model = SegformerForSemanticSegmentation.from_pretrained(model_folder_path)
    model.to(device)
    model.eval()
    print("✓ Face Parsing Model Loaded Successfully")
    print(f"  Device: {device}")
except Exception as e:
    print(f"✗ Error loading model: {e}")
    print("Please ensure the face_parsing folder is in your Google Drive")

## 5. Load Image from Google Drive

In [ ]:
try:
    image = Image.open(image_path).convert("RGB")
    img_rgb = np.array(image)
    print(f"✓ Image loaded: {image.size}")
    print(f"  Shape: {img_rgb.shape}")
    
    # Display original image
    plt.figure(figsize=(10, 8))
    plt.imshow(img_rgb)
    plt.title("Original Image", fontsize=14, fontweight="bold")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
except Exception as e:
    print(f"✗ Error loading image: {e}")
    print("Please ensure the image file exists at the specified path")

## 6. Run Face Segmentation

In [ ]:
print("Running face segmentation...")

# Run segmentation
inputs = processor(images=image, return_tensors="pt").to(device)

with torch.no_grad():
    outputs = model(**inputs)

logits = outputs.logits

# Upsample to original image size
upsampled_logits = torch.nn.functional.interpolate(
    logits,
    size=image.size[::-1],
    mode="bilinear",
    align_corners=False,
)

labels = upsampled_logits.argmax(dim=1)[0]
labels = labels.cpu().numpy()

print(f"✓ Segmentation complete")
print(f"Unique labels detected: {sorted(np.unique(labels))}")

## 7. Define Facial Part Labels

In [ ]:
LABEL_MAP = {
    0: "background",
    1: "skin",
    2: "nose",
    3: "eye_g",
    4: "left_eye",
    5: "right_eye",
    6: "left_eyebrow",
    7: "right_eyebrow",
    8: "left_ear",
    9: "right_ear",
    10: "mouth",
    11: "upper_lip",
    12: "lower_lip",
    13: "hair",
    14: "hat",
    15: "earring",
    16: "necklace",
    17: "neck",
    18: "cloth",
}

print(f"Total facial regions: {len(LABEL_MAP)}")

## 8. Extract Hair Region

In [ ]:
# Hair label ID is 13
HAIR_LABEL = 13

# Create hair mask
hair_mask = (labels == HAIR_LABEL)

# Statistics
hair_pixel_count = np.sum(hair_mask)
total_pixels = hair_mask.size
hair_coverage = (hair_pixel_count / total_pixels * 100)

print(f"\n{'='*60}")
print("HAIR EXTRACTION RESULTS")
print(f"{'='*60}")
print(f"Hair pixels detected: {hair_pixel_count:,}")
print(f"Total image pixels: {total_pixels:,}")
print(f"Hair coverage: {hair_coverage:.2f}%")
print(f"{'='*60}\n")

## 9. Visualize Hair Mask

In [ ]:
# Display hair mask
plt.figure(figsize=(12, 9))
plt.imshow(hair_mask, cmap="gray")
plt.colorbar(label="Hair Region (White=Hair, Black=Background)", shrink=0.8)
plt.title(f"Hair Segmentation Mask ({hair_coverage:.2f}% coverage)", fontsize=14, fontweight="bold")
plt.axis("off")
plt.tight_layout()
plt.show()

## 10. Hair Extraction - Method 1: White Background

In [ ]:
# Extract hair with white background
hair_white_bg = np.ones_like(img_rgb) * 255
hair_white_bg[hair_mask] = img_rgb[hair_mask]

# Display
plt.figure(figsize=(12, 9))
plt.imshow(hair_white_bg)
plt.title("Hair Extraction - White Background (Full Size)", fontsize=14, fontweight="bold")
plt.axis("off")
plt.tight_layout()
plt.show()

print("✓ Hair extracted with white background")

## 11. Hair Extraction - Method 2: Transparent Background

In [ ]:
# Extract hair with transparent background (RGBA)
hair_rgba = np.zeros((img_rgb.shape[0], img_rgb.shape[1], 4), dtype=np.uint8)
hair_rgba[hair_mask, :3] = img_rgb[hair_mask]  # RGB channels
hair_rgba[hair_mask, 3] = 255  # Alpha channel (opaque where hair)

# Display (show RGB channels)
fig, ax = plt.subplots(figsize=(12, 9))
ax.imshow(hair_rgba[:, :, :3])
ax.set_title("Hair Extraction - Transparent Background (RGB Shown)", fontsize=14, fontweight="bold")
ax.axis("off")
plt.tight_layout()
plt.show()

print("✓ Hair extracted with transparent background (RGBA format)")
print("  Note: Alpha channel encodes transparency information")

## 12. Hair Extraction - Method 3: Cropped to Bounding Box

In [ ]:
# Find bounding box
ys, xs = np.where(hair_mask)

if len(xs) > 0:
    x1, x2 = xs.min(), xs.max()
    y1, y2 = ys.min(), ys.max()
    
    print(f"Hair bounding box: ({x1}, {y1}) to ({x2}, {y2})")
    print(f"Cropped size: {x2-x1} x {y2-y1} pixels")
    print()
    
    # Crop image and mask
    hair_cropped = img_rgb[y1:y2+1, x1:x2+1].copy()
    hair_mask_cropped = hair_mask[y1:y2+1, x1:x2+1]
    
    # Add white background
    hair_cropped_white_bg = np.ones_like(hair_cropped) * 255
    hair_cropped_white_bg[hair_mask_cropped] = hair_cropped[hair_mask_cropped]
    
    # Display
    plt.figure(figsize=(12, 9))
    plt.imshow(hair_cropped_white_bg)
    plt.title(f"Hair Extraction - Cropped (Size: {hair_cropped.shape[1]}x{hair_cropped.shape[0]}px)", 
             fontsize=14, fontweight="bold")
    plt.axis("off")
    plt.tight_layout()
    plt.show()
    
    print("✓ Hair extracted and cropped to bounding box")
else:
    print("✗ No hair detected in image")
    hair_cropped_white_bg = None

## 13. Comprehensive Comparison: All Methods

In [ ]:
# Side-by-side comparison of all extraction methods
fig, axes = plt.subplots(2, 2, figsize=(18, 16))

# Original image
axes[0, 0].imshow(img_rgb)
axes[0, 0].set_title("Original Image", fontsize=13, fontweight="bold")
axes[0, 0].axis("off")

# Hair mask
axes[0, 1].imshow(hair_mask, cmap="gray")
axes[0, 1].set_title("Hair Mask (Binary)", fontsize=13, fontweight="bold")
axes[0, 1].axis("off")

# Hair with white background
axes[1, 0].imshow(hair_white_bg)
axes[1, 0].set_title("Hair - White Background (Full Size)", fontsize=13, fontweight="bold")
axes[1, 0].axis("off")

# Hair cropped
if hair_cropped_white_bg is not None:
    axes[1, 1].imshow(hair_cropped_white_bg)
    axes[1, 1].set_title("Hair - Cropped (White Background)", fontsize=13, fontweight="bold")
else:
    axes[1, 1].text(0.5, 0.5, "No hair detected", ha="center", va="center", fontsize=14)
    axes[1, 1].set_title("Hair - Cropped", fontsize=13, fontweight="bold")

axes[1, 1].axis("off")

plt.suptitle(f"Hair Extraction Comparison (Coverage: {hair_coverage:.2f}%)", 
             fontsize=15, fontweight="bold", y=0.995)
plt.tight_layout()
plt.show()

## 14. Hair Overlay Visualization

In [ ]:
# Create overlay showing detected hair region
overlay = img_rgb.copy().astype(float)

# Highlight hair region in red
overlay[hair_mask, 0] = 255  # Red channel
overlay[hair_mask, 1] = overlay[hair_mask, 1] * 0.5  # Reduce green
overlay[hair_mask, 2] = overlay[hair_mask, 2] * 0.5  # Reduce blue

overlay = overlay.astype(np.uint8)

# Blend original and overlay
blended = cv2.addWeighted(img_rgb, 0.6, overlay, 0.4, 0)

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

axes[0].imshow(img_rgb)
axes[0].set_title("Original Image", fontsize=13, fontweight="bold")
axes[0].axis("off")

axes[1].imshow(blended)
axes[1].set_title("Hair Region Highlighted (Red Overlay)", fontsize=13, fontweight="bold")
axes[1].axis("off")

plt.tight_layout()
plt.show()

## 15. All Facial Parts Detection & Coverage

In [ ]:
# Show segmentation map with all labels
fig, axes = plt.subplots(1, 2, figsize=(18, 8))

# Full segmentation map
im1 = axes[0].imshow(labels, cmap="nipy_spectral")
axes[0].set_title("Face Parsing Segmentation Map (All Parts)", fontsize=13, fontweight="bold")
axes[0].axis("off")
plt.colorbar(im1, ax=axes[0], label="Label ID")

# Show individual parts detected
detected_parts = []
for label_id in np.unique(labels):
    part_name = LABEL_MAP.get(label_id, f"Unknown {label_id}")
    pixel_count = np.sum(labels == label_id)
    percentage = (pixel_count / total_pixels * 100)
    detected_parts.append((label_id, part_name, pixel_count, percentage))

# Text display
text_str = "Detected Facial Parts:\n\n"
for label_id, part_name, pixel_count, percentage in sorted(detected_parts, key=lambda x: x[3], reverse=True):
    if percentage > 0.1:  # Only show parts with >0.1% coverage
        bar_width = int(percentage / 2)  # Scale for display
        text_str += f"{part_name:.<25} {percentage:>6.2f}% {'█' * bar_width}\n"

axes[1].text(0.05, 0.95, text_str, transform=axes[1].transAxes, fontsize=10,
             verticalalignment="top", fontfamily="monospace",
             bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.8))
axes[1].axis("off")
axes[1].set_title("Detected Parts & Coverage", fontsize=13, fontweight="bold")

plt.tight_layout()
plt.show()

## 16. Detailed Statistics & Analysis

In [ ]:
print("\n" + "="*70)
print("HAIR EXTRACTION - DETAILED STATISTICS")
print("="*70)
print(f"\nImage Dimensions:")
print(f"  Width:  {img_rgb.shape[1]} pixels")
print(f"  Height: {img_rgb.shape[0]} pixels")
print(f"  Total pixels: {total_pixels:,}")

print(f"\nHair Region:")
print(f"  Hair pixels: {hair_pixel_count:,}")
print(f"  Coverage: {hair_coverage:.2f}%")

if hair_cropped_white_bg is not None:
    print(f"\nBounding Box:")
    print(f"  Top-Left: ({x1}, {y1})")
    print(f"  Bottom-Right: ({x2}, {y2})")
    print(f"  Width: {x2-x1} pixels")
    print(f"  Height: {y2-y1} pixels")
    print(f"  Area: {(x2-x1)*(y2-y1):,} pixels")

print(f"\nExtraction Methods Available:")
print(f"  1. White Background (Full Size)")
print(f"  2. Transparent Background (RGBA)")
print(f"  3. Cropped to Bounding Box")

print(f"\nVariables Stored in Memory:")
print(f"  • hair_white_bg: Full-size hair with white background")
print(f"  • hair_rgba: Full-size hair with transparency (RGBA)")
print(f"  • hair_cropped_white_bg: Cropped hair image (if detected)")
print(f"  • hair_mask: Binary segmentation mask")
print(f"  • labels: Full segmentation labels array")
print(f"  • img_rgb: Original image array")

print("\n" + "="*70)

## 17. Summary & Next Steps

In [ ]:
print("\n" + "#"*70)
print("# HAIR EXTRACTION - SUMMARY")
print("#"*70)
print(f"\n✓ Successfully extracted hair region from image")
print(f"\nSetup:")
print(f"  Model: Face Parsing (SegFormer) from Google Drive")
print(f"  Image: Loaded from Google Drive")
print(f"\nResults:")
print(f"  Hair pixels detected: {hair_pixel_count:,}")
print(f"  Image coverage: {hair_coverage:.2f}%")
print(f"\nOutput Variables (in memory):")
print(f"  • hair_white_bg: Full-size with white background")
print(f"  • hair_rgba: Full-size with transparency (RGBA)")
print(f"  • hair_cropped_white_bg: Cropped version (if hair detected)")
print(f"  • hair_mask: Binary segmentation mask")
print(f"\n✓ All visualizations displayed above")
print(f"\nNote: Images are stored in memory, not saved to disk")
print(f"      (To save images, use cv2.imwrite() or Image.save())")
print("\n" + "#"*70)

## 18. Hair Color Analysis

In [ ]:
from sklearn.cluster import KMeans
from collections import Counter

print("Analyzing hair colors...\n")

# Extract hair pixels from original image
hair_pixels = img_rgb[hair_mask]

print(f"Total hair pixels for analysis: {len(hair_pixels):,}")

# Use KMeans to find dominant colors
n_colors = 5  # Number of dominant colors to extract
kmeans = KMeans(n_clusters=n_colors, random_state=42, n_init=10)
kmeans.fit(hair_pixels)

# Get cluster centers (dominant colors)
dominant_colors = kmeans.cluster_centers_.astype(int)
labels_kmeans = kmeans.labels_

# Get color distribution
color_distribution = Counter(labels_kmeans)
total_hair_pixels = len(labels_kmeans)

# Sort by frequency
sorted_colors = sorted(color_distribution.items(), key=lambda x: x[1], reverse=True)

print(f"\n{'='*60}")
print("DOMINANT HAIR COLORS")
print(f"{'='*60}")

color_info = []
for idx, (color_idx, count) in enumerate(sorted_colors, 1):
    rgb = dominant_colors[color_idx]
    hex_color = '#{:02x}{:02x}{:02x}'.format(rgb[0], rgb[1], rgb[2])
    percentage = (count / total_hair_pixels) * 100
    color_info.append({
        'rank': idx,
        'rgb': rgb,
        'hex': hex_color,
        'percentage': percentage,
        'count': count
    })
    print(f"{idx}. {hex_color} - RGB{tuple(rgb)} - {percentage:.2f}% ({count:,} pixels)")

print(f"{'='*60}\n")

## 19. Hair Color Palette Visualization

In [ ]:
# Create color palette visualization
fig, axes = plt.subplots(2, 2, figsize=(16, 12))

# 1. Hair image
axes[0, 0].imshow(hair_white_bg)
axes[0, 0].set_title("Extracted Hair", fontsize=13, fontweight="bold")
axes[0, 0].axis("off")

# 2. Dominant colors palette (horizontal bars)
ax_palette = axes[0, 1]
bar_height = 1
y_pos = 0

for info in color_info:
    color_normalized = [c/255.0 for c in info['rgb']]
    ax_palette.barh(y_pos, info['percentage'], height=0.8, color=color_normalized, 
                     edgecolor='black', linewidth=2)
    ax_palette.text(-5, y_pos, f"{info['hex']}", va='center', ha='right', fontsize=11, fontweight='bold')
    ax_palette.text(info['percentage'] + 2, y_pos, f"{info['percentage']:.1f}%", 
                   va='center', ha='left', fontsize=10)
    y_pos += 1

ax_palette.set_xlim(-15, 105)
ax_palette.set_ylim(-0.5, len(color_info) - 0.5)
ax_palette.set_yticks(range(len(color_info)))
ax_palette.set_yticklabels([f"Color {i+1}" for i in range(len(color_info))])
ax_palette.set_xlabel('Percentage (%)', fontsize=11, fontweight='bold')
ax_palette.set_title('Hair Color Distribution', fontsize=13, fontweight='bold')
ax_palette.grid(axis='x', alpha=0.3)

# 3. Color swatches (large)
ax_swatches = axes[1, 0]
swatch_width = 100
swatch_height = 50

swatch_image = np.zeros((len(color_info) * swatch_height, swatch_width * len(color_info), 3), dtype=np.uint8)

for idx, info in enumerate(color_info):
    y_start = idx * swatch_height
    swatch_image[y_start:y_start + swatch_height, :] = info['rgb']

ax_swatches.imshow(swatch_image)
ax_swatches.set_title('Color Swatches (Top to Bottom)', fontsize=13, fontweight='bold')
ax_swatches.set_xticks([])
ax_swatches.set_yticks([i * swatch_height + swatch_height//2 for i in range(len(color_info))])
ax_swatches.set_yticklabels([info['hex'] for info in color_info])

# 4. Color information table
ax_table = axes[1, 1]
ax_table.axis('off')

table_data = []
table_data.append(['Rank', 'Hex Color', 'RGB', 'Coverage'])
for info in color_info:
    table_data.append([
        str(info['rank']),
        info['hex'],
        f"({info['rgb'][0]}, {info['rgb'][1]}, {info['rgb'][2]})",
        f"{info['percentage']:.1f}%"
    ])

table = ax_table.table(cellText=table_data, cellLoc='center', loc='center',
                       colWidths=[0.12, 0.25, 0.35, 0.18])
table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 2.5)

# Style header
for i in range(4):
    table[(0, i)].set_facecolor('#40466e')
    table[(0, i)].set_text_props(weight='bold', color='white')

# Color code rows by color
for idx, info in enumerate(color_info, 1):
    color_normalized = [c/255.0 for c in info['rgb']]
    table[(idx, 0)].set_facecolor('#f0f0f0')
    for j in range(1, 4):
        table[(idx, j)].set_facecolor(color_normalized)
        # Set text color based on luminance
        luminance = (0.299 * info['rgb'][0] + 0.587 * info['rgb'][1] + 0.114 * info['rgb'][2]) / 255
        text_color = 'white' if luminance < 0.5 else 'black'
        table[(idx, j)].set_text_props(color=text_color, weight='bold')

ax_table.set_title('Hair Color Details', fontsize=13, fontweight='bold', pad=20)

plt.suptitle(f'Hair Color Analysis ({hair_coverage:.2f}% coverage)', 
             fontsize=15, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

## 20. Hair Color Category Classification

In [ ]:
def classify_hair_color(hex_color):
    """
    Classify hair color into categories based on hex color.
    """
    hex_color = hex_color.lstrip('#')
    r, g, b = int(hex_color[0:2], 16), int(hex_color[2:4], 16), int(hex_color[4:6], 16)
    
    # Calculate brightness
    brightness = (r + g + b) / 3
    
    # Calculate color dominance
    if r > g and r > b:
        if brightness > 200:
            return "Light/Golden Blonde"
        elif brightness > 150:
            return "Dark Blonde/Honey"
        elif brightness > 100:
            return "Light Brown/Chestnut"
        elif brightness > 50:
            return "Dark Brown"
        else:
            return "Black/Very Dark Brown"
    elif g > r and g > b:
        return "Green/Dyed Hair"
    elif b > r and b > g:
        return "Blue/Dyed Hair"
    else:
        # Neutral - use brightness as tiebreaker
        if brightness > 200:
            return "Platinum/Light Blonde"
        elif brightness > 150:
            return "Light Brown/Ash Blonde"
        elif brightness > 100:
            return "Medium Brown"
        elif brightness > 50:
            return "Dark Brown"
        else:
            return "Black/Very Dark Brown"

# Classify primary hair color
primary_color = color_info[0]
primary_hex = primary_color['hex']
hair_category = classify_hair_color(primary_hex)

print(f"\n{'='*60}")
print("HAIR COLOR CLASSIFICATION")
print(f"{'='*60}")
print(f"\nPrimary Hair Color: {primary_hex}")
print(f"Hair Category: {hair_category}")
print(f"Coverage: {primary_color['percentage']:.2f}%")
print(f"\nAll Color Categories:")

for info in color_info:
    category = classify_hair_color(info['hex'])
    print(f"  {info['hex']} ({info['percentage']:.1f}%) → {category}")

print(f"{'='*60}\n")

## 21. Hair Color Summary Report

In [ ]:
# Create comprehensive hair color report
fig = plt.figure(figsize=(16, 10))
gs = fig.add_gridspec(2, 2, hspace=0.35, wspace=0.3)

# 1. Hair image with color overlay
ax1 = fig.add_subplot(gs[0, :])
ax1.imshow(hair_white_bg)
ax1.set_title(f"Your Hair Color: {hair_category}", fontsize=14, fontweight="bold")
ax1.axis("off")

# 2. Color palette strip
ax2 = fig.add_subplot(gs[1, 0])
palette_strip = np.zeros((100, 500, 3), dtype=np.uint8)

segment_width = 500 // len(color_info)
for idx, info in enumerate(color_info):
    x_start = idx * segment_width
    x_end = (idx + 1) * segment_width
    palette_strip[:, x_start:x_end] = info['rgb']

ax2.imshow(palette_strip)
ax2.set_title("Shades of Your Hair", fontsize=12, fontweight="bold")
ax2.set_xticks([])
ax2.set_yticks([])
ax2.spines['top'].set_visible(False)
ax2.spines['right'].set_visible(False)
ax2.spines['bottom'].set_visible(False)
ax2.spines['left'].set_visible(False)

# 3. Description and info
ax3 = fig.add_subplot(gs[1, 1])
ax3.axis('off')

# Create descriptive text
description = f"""Hair Color Analysis Report

"""
description += f"Primary Color: {primary_hex}\n"
description += f"Classification: {hair_category}\n"
description += f"Coverage: {primary_color['percentage']:.1f}%\n\n"

description += "Color Breakdown:\n"
for idx, info in enumerate(color_info, 1):
    description += f"  {idx}. {info['hex']} - {info['percentage']:.1f}%\n"

description += f"\nTotal Hair Pixels: {total_hair_pixels:,}\n"
description += f"Image Coverage: {hair_coverage:.2f}%"

ax3.text(0.05, 0.95, description, transform=ax3.transAxes, fontsize=11,
         verticalalignment='top', fontfamily='monospace',
         bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.8, pad=1))

plt.suptitle("Hair Color Summary", fontsize=16, fontweight="bold", y=0.98)
plt.show()

## 23. Hairline Shape Detection - Landmark Extraction

In [ ]:
from scipy.signal import savgol_filter
import cv2

print("Detecting hairline landmarks...\n")

skin_mask = (labels == 1)
eyebrow_mask = (labels == 6) | (labels == 7)
eye_mask = (labels == 4) | (labels == 5)

if eyebrow_mask.sum() > 0:
    eye_level_y = int(np.mean(np.where(eyebrow_mask)[0]))
elif eye_mask.sum() > 0:
    eye_level_y = int(np.mean(np.where(eye_mask)[0]))
else:
    eye_level_y = int(img_rgb.shape[0] * 0.4)

top_of_head_y = int(np.min(np.where(hair_mask)[0])) if hair_mask.sum() > 0 else 0
forehead_height = max(eye_level_y - top_of_head_y, 1)

temple_row = int(top_of_head_y + 0.6 * forehead_height)
temple_row = min(max(temple_row, 0), img_rgb.shape[0] - 1)

face_row_mask = skin_mask[temple_row] | hair_mask[temple_row]
xs_row = np.where(face_row_mask)[0]

if len(xs_row) > 0:
    temple_left_x, temple_right_x = int(xs_row.min()), int(xs_row.max())
else:
    ys_h, xs_h = np.where(hair_mask)
    temple_left_x, temple_right_x = int(xs_h.min()), int(xs_h.max())

x_range = np.arange(temple_left_x, temple_right_x + 1)
hairline_y_raw = np.full(len(x_range), np.nan)

for i, x in enumerate(x_range):
    col = hair_mask[top_of_head_y:eye_level_y, x]
    ys_col = np.where(col)[0]
    if len(ys_col) > 0:
        hairline_y_raw[i] = top_of_head_y + ys_col.max()

valid = ~np.isnan(hairline_y_raw)
if valid.sum() >= 2:
    hairline_y_raw[~valid] = np.interp(x_range[~valid], x_range[valid], hairline_y_raw[valid])
else:
    hairline_y_raw[~valid] = eye_level_y

n_pts = len(hairline_y_raw)
window = min(31, n_pts if n_pts % 2 == 1 else n_pts - 1)
window = max(window, 5)
if window % 2 == 0:
    window -= 1
polyorder = 3 if window > 3 else 1
hairline_y_smooth = savgol_filter(hairline_y_raw, window_length=window, polyorder=polyorder)

n = len(x_range)
idx_A, idx_B, idx_C, idx_D, idx_E = 0, int(n*0.25), int(n*0.5), int(n*0.75), n-1

landmarks = {
    'A': (int(x_range[idx_A]), float(hairline_y_smooth[idx_A])),
    'B': (int(x_range[idx_B]), float(hairline_y_smooth[idx_B])),
    'C': (int(x_range[idx_C]), float(hairline_y_smooth[idx_C])),
    'D': (int(x_range[idx_D]), float(hairline_y_smooth[idx_D])),
    'E': (int(x_range[idx_E]), float(hairline_y_smooth[idx_E])),
}

print(f"Face width: {temple_left_x} to {temple_right_x} ({temple_right_x-temple_left_x}px)")
print(f"Forehead height (top-of-head to eye level): {forehead_height}px\n")
print("Landmark points (x, y):")
for name, (px, py) in landmarks.items():
    print(f"  {name}: ({px}, {py:.1f})")

## 24. Hairline Shape Visualization with Landmarks

In [ ]:
import matplotlib.pyplot as plt

# --------------------------------------------
# Only original image with small landmark circles
# --------------------------------------------

fig, ax = plt.subplots(figsize=(7, 7))

ax.imshow(img_rgb)

# Draw only small circles (no A/B/C/D text)
for _, (px, py) in landmarks.items():

    ax.scatter(
        px,
        py,
        s=22,                 # smaller circle
        facecolors="white",
        edgecolors="black",
        linewidths=1.0,
        zorder=5
    )

ax.set_title("Hairline Landmarks", fontsize=14, fontweight="bold")
ax.axis("off")

plt.tight_layout()
plt.show()

## 25. Hairline Shape Classification

In [ ]:
def normalize(v):
    return v / forehead_height

temple_avg_y = (landmarks['A'][1] + landmarks['E'][1]) / 2
center_y = landmarks['C'][1]
recession_score = normalize(center_y - temple_avg_y)

if recession_score < 0.12:
    temple_recession = "Minimal"
elif recession_score < 0.28:
    temple_recession = "Moderate"
else:
    temple_recession = "Significant"

side_avg_y = (landmarks['B'][1] + landmarks['D'][1]) / 2
widow_score = normalize(center_y - side_avg_y)
widows_peak = "Present" if widow_score > 0.08 else "Absence"

def segment_curvature(x_pts, y_pts):
    if len(x_pts) < 3:
        return 0.0
    dy = np.gradient(y_pts, x_pts)
    d2y = np.gradient(dy, x_pts)
    return float(np.mean(np.abs(d2y)))

left_seg = (x_range >= landmarks['A'][0]) & (x_range <= landmarks['B'][0])
right_seg = (x_range >= landmarks['D'][0]) & (x_range <= landmarks['E'][0])
lateral_curv = (segment_curvature(x_range[left_seg], hairline_y_smooth[left_seg]) +
                segment_curvature(x_range[right_seg], hairline_y_smooth[right_seg])) / 2

if lateral_curv < 0.01:
    lateral_shape = "Straight"
elif lateral_curv < 0.05:
    lateral_shape = "Slightly Rounded"
elif lateral_curv < 0.15:
    lateral_shape = "Rounded"
else:
    lateral_shape = "Angular"

central_mask = (x_range >= x_range[int(n*0.3)]) & (x_range <= x_range[int(n*0.7)])
central_std = normalize(np.std(hairline_y_smooth[central_mask]))

if central_std < 0.03:
    frontal_eminence = "Flat"
elif central_std < 0.08:
    frontal_eminence = "Slightly Rounded"
else:
    frontal_eminence = "Prominent"

if widows_peak == "Present" and temple_recession in ["Moderate", "Significant"]:
    overall_shape = "M-Shaped"
elif widows_peak == "Present":
    overall_shape = "Widow's Peak"
elif temple_recession == "Significant":
    overall_shape = "Receding"
elif lateral_shape in ["Rounded", "Slightly Rounded"] and frontal_eminence in ["Flat", "Slightly Rounded"]:
    overall_shape = "Rounded"
elif lateral_shape == "Straight" and frontal_eminence == "Flat":
    overall_shape = "Straight"
else:
    overall_shape = "Rounded"

print(f"{'='*60}")
print("HAIRLINE SHAPE CLASSIFICATION")
print(f"{'='*60}")
print(f"Overall Shape:      {overall_shape}")
print(f"Temple Recession:   {temple_recession}")
print(f"Widow's Peak:       {widows_peak}")
print(f"Lateral Shape:      {lateral_shape}")
print(f"Frontal Eminence:   {frontal_eminence}")
print(f"{'='*60}")